# 🌐 Modelos de Lenguaje

**Laboratorio de PLN — IFTS24**
Matías Barreto, 2026

**Encuentro 13 · Bloque 1 — 50 minutos**

---

## Objetivo

Ejecutar un modelo de lenguaje local para entender cómo los LLMs generan texto.

## Al terminar este bloque vas a poder:

1. Cargar y ejecutar Phi-3 en Google Colab usando Hugging Face.
2. Configurar parámetros de inferencia para controlar la salida del modelo.
3. Estructurar prompts en español para distintas tareas.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **LLM (Large Language Model)** | Red neuronal entrenada con enormes cantidades de texto para predecir y generar lenguaje natural. |
| **Inferencia** | El momento en que el modelo ya entrenado recibe un prompt y produce una respuesta. |
| **Phi-3** | Modelo compacto de Microsoft (3.8B parámetros) optimizado para correr en hardware limitado. |
| **Hugging Face Hub** | Repositorio online con miles de modelos listos para descargar y usar. |
| **Tokenizador** | Componente que convierte texto en números que el modelo puede procesar. |


## ⚙️ Configuración del entorno

### Requisitos

| Requisito | Detale |
|---|---|
| **Google Colab con GPU** | `Entorno de ejecución → Cambiar tipo → GPU T4` |
| **Conexión a Internet** | Para descargar el modelo desde Hugging Face Hub |

La GPU T4 gratuita de Colab es suficiente para Phi-3-mini.

## Instalación de dependencias

- **torch**: biblioteca de PyTorch con soporte CUDA para aceleración GPU
- **transformers**: biblioteca de Hugging Face para trabajar con LLMs
- **accelerate**: optimiza la carga e inferencia de modelos grandes

In [1]:
# Instalación de dependencias
# El prefijo ! permite ejecutar comandos de sistema desde el notebook
# Instalamos PyTorch con soporte CUDA
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers>=4.40.1 accelerate>=0.27.2

Looking in indexes: https://download.pytorch.org/whl/cu118



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## ¿Qué es un modelo de lenguaje?

### Analogía

Imaginá un empleado que leyó millones de libros, correos, artículos y conversaciones. Nunca entendió cada texto en profundidad, pero absorbió tantos patrones que puede predecir con alta precisión qué palabra viene después de cualquier secuencia. Eso hace un LLM: predice la siguiente palabra, una y otra vez, hasta generar texto coherente.

### Dónde vive esto en el mundo real

Los LLMs son el motor de ChatGPT, Gemini, Claude y Copilot. Las empresas los usan para atención al cliente, redacción de documentos, análisis de contratos y generación de código. En este curso, Phi-3 va a ser tu primer LLM ejecutable en Colab. En el próximo bloque vas a ver Ollama, que te permite correrlos sin conexión a Internet y sin costo de API.

### ¿Qué es Phi-3?

| Característica | Detale |
|---|---|
| **Parámetros** | 3.8 mil millones |
| **Contexto máximo** | 4096 tokens (~3000 palabras) |
| **Tipo** | Instruction-tuned (sigue instrucciones) |
| **Idiomas** | Multilingüe, incluye español |
| **Licencia** | Open source (Microsoft) |

### ¿Cómo genera texto?

El modelo opera en tres pasos:

1. **Entrada**: recibe un prompt (texto inicial)
2. **Procesamiento**: analiza el contexto usando su conocimiento interno
3. **Generación**: produce la continuación más probable, token por token

### ✎ Para pensar

- Si el modelo predice la siguiente palabra basándose en patrones estadísticos, ¿por qué puede "inventar" datos que no existen?
- ¿Qué diferencia hay entre un modelo que "sabe" algo y uno que "aprendió a responder" sobre ese algo?

## Carga del modelo

Vamos a cargar Phi-3 en la memoria de la GPU. Este proceso puede tardar 1-2 minutos la primera vez.

In [4]:
# Verificar disponibilidad de CUDA antes de cargar el modelo
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA not available. The model will run on CPU, which will be much slower.")
    print("To enable GPU in Google Colab: Runtime -> Change runtime type -> GPU T4")

PyTorch version: 2.7.1+cu118
CUDA available: False
To enable GPU in Google Colab: Runtime -> Change runtime type -> GPU T4


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Cargamos el modelo Phi-3-mini
# Este proceso puede tardar 1-2 minutos la primera vez
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",  # Identificador del modelo en Hugging Face
    device_map="cuda",                    # device_map: Dónde cargar el modelo ("cuda" = GPU, "cpu" = procesador)
    torch_dtype="auto",                   # torch_dtype: Tipo de datos numéricos ("auto" selecciona el óptimo)
    trust_remote_code=False,              # trust_remote_code: Por seguridad, no ejecutar código remoto no verificado
)

# Cargamos el tokenizador correspondiente al modelo
# El tokenizador convierte texto en números que el modelo puede procesar
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

print("Modelo cargado exitosamente")

KeyboardInterrupt: 

### Componentes cargados

| Componente | Para qué sirve |
|---|---|
| `AutoModelForCausalLM` | Carga el modelo generativo (predice tokens de izquierda a derecha) |
| `AutoTokenizer` | Convierte texto en números y números en texto |
| `device_map="cuda"` | Carga el modelo en la GPU (mucho más rápido que CPU) |
| `torch_dtype="auto"` | Selecciona automáticamente la precisión numérica óptima |


## Pipeline de generación

Un **pipeline** encapsula en un objeto todos los pasos: tokenización → inferencia → decodificación. Lo usás como una función.

In [ ]:
from transformers import pipeline

# Creamos un pipeline de generación de texto
generador = pipeline(
    "text-generation",           # Tipo de tarea
    model=model,                  # Modelo a usar
    tokenizer=tokenizer,          # Tokenizador a usar
    return_full_text=False,       # return_full_text: Si False, solo devuelve el texto generado (no el prompt)
    max_new_tokens=500,           # max_new_tokens: Cantidad máxima de tokens a generar
    do_sample=False               # do_sample: Si False, usa generación determinística (siempre la misma salida)
)

print("Pipeline creado exitosamente")

### Parámetros clave del pipeline

| Parámetro | Efecto |
|---|---|
| `return_full_text=False` | Devuelve solo el texto generado (sin repetir el prompt) |
| `max_new_tokens=500` | Límite de tokens a generar (más tokens = más tiempo) |
| `do_sample=False` | Generación determinística: siempre elige el token más probable |


### ✎ Para pensar

- Con `do_sample=False`, el modelo siempre da la misma respuesta al mismo prompt. ¿En qué casos preferirías eso? ¿En cuáles no?
- ¿Por qué tiene sentido que `max_new_tokens` afecte el tiempo de ejecución?

## Primer ejemplo: generación de texto

El modelo espera mensajes en formato de conversación estructurada:

```python
[{"role": "user", "content": "tu instrucción"}]
```

- `role`: quién habla ("user" o "assistant")
- `content`: el texto del mensaje

Este formato permite simular conversaciones de múltiples turnos.

In [ ]:
# Definimos el mensaje como una conversación
# Phi-3 está entrenado para recibir mensajes en formato de conversación
mensaje = [
    {"role": "user", "content": "Escribí un párrafo corto sobre el dulce de leche y su importancia en la cultura argentina."}
]

# Generamos la respuesta
salida = generador(mensaje)

# Mostramos el resultado
print(salida[0]["generated_text"])

## Prompt engineering: técnicas básicas

| Técnica | Ejemplo |
|---|---|
| **Sé específico** | "Escribí un email formal..." en lugar de "hablame del mate" |
| **Especificá el formato** | "Listá 5 ventajas usando viñetas" |
| **Incluí contexto** | "Para una audiencia técnica, explicá..." |
| **Pedí extensión** | "En no más de 3 párrafos..." |


### ✎ Para pensar

- ¿Por qué un prompt más específico genera mejores respuestas? ¿Qué le estás dando al modelo que no tenía antes?
- Probá el mismo prompt con `do_sample=True` y `temperature=0.9`. ¿Qué cambia?

In [ ]:
# ─── Espacio de práctica ──────────────────────────────────────────────────────
#
# Probá los siguientes prompts y comparalos con tus compañeros:
#   1. Un texto formal (carta, informe, email académico)
#   2. Un texto creativo (cuento corto, poema, diálogo)
#   3. Un texto técnico (explicación de un concepto, tutorial de un paso)
#
# Bonus: ¿cómo cambia la respuesta si especificás la longitud o el tono?
#
# ─────────────────────────────────────────────────────────────────────────────
mi_prompt = [
    {"role": "user", "content": "Tu instrucción aquí"}
]

mi_salida = generador(mi_prompt)
print(mi_salida[0]["generated_text"])

## ⛰️ Cierre del bloque

| Concepto | Qué aprendiste |
|---|---|
| **LLM** | Red neuronal que predice tokens para generar texto coherente |
| **Phi-3** | Modelo compacto (3.8B parámetros) ejecutable en Colab con GPU |
| **Pipeline** | Objeto que encapsula tokenización + inferencia + decodificación |
| **Parámetros** | `max_new_tokens`, `do_sample`, `temperature` controlan la salida |
| **Prompt** | La instrucción en formato `{"role": "user", "content": "..."}` |

**Próximo bloque →** Tokens y Embeddings: cómo el modelo "lee" el texto antes de generarlo, y por qué ese proceso determina qué tan bien entiende el español.